In [1]:
import pandas as pd
import os
from google.colab import drive
import duckdb
drive.mount('/content/drive')
path =  '/content/drive/MyDrive/Psychologia/Magisterka/magisterka Psyche/Mateusz_Kamyczura_Praca_Magisterska/Praca Magisterska/Kody i Dane/Baza danych'
mABC_path = path + '/metryczkaABC'
mXYZ_path = path + '/metryczkaXYZ'
tABC_path = path + '/tabeleABC'
tXYZ_path = path + '/tabeleXYZ'
Klucz_path = path + '/Input'
Output_path = path + '/Output'
poprawneABC_path = Klucz_path + '/kluczABC.xlsx'
ppoprawneXYZ_path = Klucz_path + '/kluczXYZ.xlsx'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
con = duckdb.connect('baza.duckdb')

##Wczytanie i połączenie danych


In [3]:
poprawneABC = pd.read_excel(Klucz_path + '/kluczABC.xlsx')
poprawneXYZ = pd.read_excel(Klucz_path + '/kluczXYZ.xlsx')

In [4]:
AGL_A2 = ["abccacb", "bacbcab", "abababcccab", "ababcabccab",
                  "abcabcabcabcab", "abcababcab", "acbcabcab", "abcbcabccbcab",
                  "abcabccab", "abcaabcaabcab", "aabcacbcaba", "ababcccababab",
                  "abccababccab", "abcabccabcb", "abcabab", "abababababab", "abcacaab",
                  "cabbbca", "accbacbab", "abcabcccab", "babcabcabccab", "ababab",
                  "abcccab", "abababab", "abbcab", "abccab", "baccb", "abababccabcab",
                  "abcabbab", "ababcabcab", ]

ALG_B2 = ["XYXYYYYYYYYXYZ", "XXXXZZZXY", "XXXXYZZZZXY", "XZZZXXXY", "XZXYZZZXY",
                  "XXYXXYXZZZZZZXY", "XXXXXXYYZZZZZXY", "XXYXYXZZZZZXY", "XXXXXXXZZZZZZXY",
                  "XYYXXZZZZZXY", "XYYYYZZZZXY", "XXXXXYYYZXZZXY", "XYXXYYZZZZZXY", "XYZXY",
                  "XYZYYYZZXYZ", "XYXYXYZYZZZXY", "XYYYZZZZZXY", "XXXYYXYZZZZZZXY", "XXXXYYZXYZZZZZY",
                  "XXZXY", "XXYZZXY", "XXYXZZZXY", "XYYXYXYZZZZZZXY", "XZZZXY", "YYYZZZXY", "XYXXXZZXXXZYXY",
                  "ZZXYXZZZXY", "XYXXZZZXY", "XXZZXY", "XXXZZXY", ]


In [5]:
poprawneABC['Ciąg'] = AGL_A2
poprawneXYZ['Ciąg'] = ALG_B2

In [6]:
def polacz_metryczki(path):
    # Funkcja łączy metryczki z folderu w jednego dużego DF
    dataframes = []

    # Przechodzenie przez wszystkie pliki w folderze
    for file in os.listdir(path):
        full_path = os.path.join(path, file)

        # Sprawdzenie rozszerzenia pliku i wczytanie go do Pandas
        if file.endswith(".csv"):
            df = pd.read_csv(full_path, index_col=0)  # Pierwsza kolumna jako indeks
            dataframes.append(df)
        elif file.endswith(".xlsx") or file.endswith(".xls"):
            df = pd.read_excel(full_path, index_col=0)  # Pierwsza kolumna jako indeks
            dataframes.append(df)

    # Jeśli nie znaleziono plików, zwracamy pusty DataFrame
    if not dataframes:
        return pd.DataFrame()

    # Konkatenujemy wszystkie DataFrame'y w jeden
    df_final = pd.concat(dataframes, ignore_index=False)  # Zachowujemy indeksy

    # Resetujemy indeks
    df_final.reset_index(drop=True, inplace=True)

    return df_final

In [7]:
metryczkiABC = polacz_metryczki(mABC_path)
metryczkiXYZ = polacz_metryczki(mXYZ_path)

In [8]:
metryczkiABC.head()

,Zgoda,Kobieta,Wiek,Wykształcenie,Zamieszkanie,inz,mat,Key
0,1,0,57,ś,w,0,0,Do1
1,1,1,55,ś,do50,0,0,Do2
2,1,1,55,ś,do50,0,0,Do3
3,1,0,26,w,500+,1,0,A1
4,1,1,23,w,500+,1,0,A2


In [9]:
def polacz_tabele(path: str) -> pd.DataFrame:
    """
    Funkcja łączy wszystkie arkusze z plików Excel w folderze 'path', dodając kolumnę 'Key'
    z nazwą arkusza, a następnie scala wszystkie arkusze w jedną tabelę.

    :param path: Ścieżka do folderu zawierającego pliki Excel
    :return: Połączony DataFrame
    """
    all_data = []

    for file in os.listdir(path):
        if file.endswith(".xlsx") or file.endswith(".xls"):
            file_path = os.path.join(path, file)
            xls = pd.ExcelFile(file_path)

            for sheet_name in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name=sheet_name)
                df["Key"] = sheet_name  # Dodaj kolumnę z nazwą arkusza
                all_data.append(df)

    merged_df = pd.concat(all_data, ignore_index=True)
    return merged_df

In [10]:
tabeleXYZ = polacz_tabele(tXYZ_path)
tabeleABC = polacz_tabele(tABC_path)
con.register("TEMPtabele", tabeleABC.reset_index())

In [11]:
tabele = con.execute("""
    CREATE OR REPLACE TABLE tabele AS
    SELECT index AS id, "Lp." as Lp, OSAD, "I/Z" AS IZ, Key
    FROM TEMPtabele
    WHERE "Lp." NOT IN (4,23)
""").df()


In [12]:
tabeleABC = tabeleABC[~tabeleABC["Lp."].isin([4, 23])]

##Odsianie obserwacji odstających

In [13]:
def sprawdz_poprawnosc_odpowiedzi(tabele: pd.DataFrame, poprawne: pd.DataFrame) -> pd.DataFrame:
    """
    Funkcja łączy dwa DataFrame'y po kolumnie "Lp" i zwraca połączoną tabelę.
    Dodatkowo dodaje kolumny 'CzyPoprawna' oraz 'obróconyOSAD'.
    CzyPoprawna mówi o tym, czy dla danego badanego i danego pytania odpowiedź była poprawna
    :param tabele: Pierwszy DataFrame
    :param poprawne: Drugi DataFrame
    :return: Połączony DataFrame
    """
    merged_df = tabele.merge(poprawne, on="Lp.", how="inner")

    # Dodaj kolumnę 'CzyPoprawna'
    merged_df["CzyPoprawna"] = merged_df.apply(lambda row: 1 if (row["OSAD"] > 5 and row["G"] == 1) or (row["OSAD"] < 5 and row["G"] == 0)
                                                else 0 if row["OSAD"] == 5
                                                else -1, axis=1)

    # Dodaj kolumnę 'obróconyOSAD'
    merged_df["obróconyOSAD"] = merged_df.apply(lambda row: row["OSAD"] if row["G"] == 1 else 10 - row["OSAD"], axis=1)

    return merged_df


def policz_procenty(df):
    """
    Liczy procent poprawnych odpowiedzi i procent odpowiedzi zgodnych z zasadą (I/Z == 2),
    oraz dodaje kolumny 'mat', 'Wiek' i 'Kobieta' – sprawdzając ich spójność.

    Argumenty:
    df : pandas.DataFrame
        Dane z kolumnami: 'Key', 'CzyPoprawna', 'I/Z', 'mat', 'Wiek', 'Kobieta'

    Zwraca:
    pandas.DataFrame z kolumnami:
        'Key', 'ProcentPoprawnych', 'ProcentZasad', 'mat', 'Wiek', 'Kobieta'
    """

    # Filtrowanie odpowiedzi różnej od 0 (np. brak odpowiedzi)
    df_filtered = df[df["CzyPoprawna"] != 0]

    # Procent poprawnych odpowiedzi
    procent_poprawnych = df_filtered.groupby("Key")["CzyPoprawna"].apply(lambda x: (x == 1).mean() * 100)

    # Procent zasad (I/Z == 2)
    procent_zasad = df.groupby("Key")["I/Z"].apply(lambda x: (x == 2).mean() * 100)

    # Sprawdzamy, czy kolumna 'mat', 'Wiek' i 'Kobieta' mają unikalną wartość dla każdego Key
    for col in ['mat', 'Wiek', 'Kobieta']:
        values_per_key = df.groupby("Key")[col].nunique()
        if (values_per_key > 1).any():
            raise ValueError(f"Dla niektórych 'Key' kolumna '{col}' zawiera różne wartości!")

    # Pobieramy wartość 'mat', 'Wiek' i 'Kobieta' dla każdego Key (po jednej reprezentatywnej wartości)
    mat_per_key = df.groupby("Key")["mat"].first()
    wiek_per_key = df.groupby("Key")["Wiek"].first()
    kobieta_per_key = df.groupby("Key")["Kobieta"].first()

    # Łączymy wszystko
    wynik = pd.concat([procent_poprawnych, procent_zasad, mat_per_key, wiek_per_key, kobieta_per_key], axis=1).reset_index()
    wynik.columns = ["Key", "ProcentPoprawnych", "ProcentZasad", "mat", "Wiek", "Kobieta"]

    return wynik

def znajdz_odstajace_IQR(df):
    """
    Zwraca listę wartości 'Key' dla obserwacji odstających metodą IQR (1.5 * IQR).

    Argumenty:
    df : pandas.DataFrame
        Tabela z danymi, gdzie:
        - "Key" to indeksy ludzi
        - "ProcentPoprawnych" to wynik procentowy

    Zwraca:
    list : lista wartości 'Key' dla obserwacji odstających
    """

    # Kopia danych
    df = df.copy()

    # Obliczamy kwartyle i IQR
    Q1 = df['ProcentPoprawnych'].quantile(0.25)
    Q3 = df['ProcentPoprawnych'].quantile(0.75)
    IQR = Q3 - Q1

    # Granice dla wartości odstających
    dolna_granica = Q1 - 1.5 * IQR
    gorna_granica = Q3 + 1.5 * IQR

    # Wyszukujemy wartości odstające
    maska_odstajace = (df['ProcentPoprawnych'] < dolna_granica) | (df['ProcentPoprawnych'] > gorna_granica)
    usuniete = df.loc[maska_odstajace, 'Key'].tolist()

    print(f"Metoda odfiltrowania: IQR")
    if usuniete:
        print(f"Znalezione obserwacje odstające (Key): {usuniete}")
    else:
        print("Nie znaleziono obserwacji odstających.")

    return usuniete


In [14]:
con.execute("""
    SELECT *
    FROM tabele
""").df()

,id,Lp,OSAD,IZ,Key
0,0,1,2,1,B1
1,1,2,1,1,B1
2,2,3,4,1,B1
3,4,5,2,1,B1
4,5,6,7,1,B1
...,...,...,...,...,...
1199,1285,26,8,2,O6
1200,1286,27,2,2,O6
1201,1287,28,6,2,O6
1202,1288,29,6,2,O6


In [15]:
poprawneABC

,Lp.,G,Ciąg
0,1,0,abccacb
1,2,0,bacbcab
2,3,1,abababcccab
3,4,0,ababcabccab
4,5,1,abcabcabcabcab
5,6,1,abcababcab
6,7,0,acbcabcab
7,8,0,abcbcabccbcab
8,9,1,abcabccab
9,10,0,abcaabcaabcab


In [16]:
con.execute("""
    CREATE OR REPLACE TABLE tabeleUZUP AS
    SELECT t.id, t.Lp, t.OSAD, t.IZ, t.Key, p.G,
    CASE
      WHEN p.G = 1 AND t.OSAD >5 THEN 1
      WHEN p.G = 0 AND t.OSAD <5 THEN 1
      WHEN t.OSAD = 5 THEN 0
      ELSE -1
    END AS CzyPoprawna,
    10 - t.OSAD AS ObroconyOSAD
    FROM tabele AS t
    JOIN poprawneABC AS p ON t."Lp" = p."Lp."
""")

In [17]:
con.execute("""
    CREATE OR REPLACE TABLE tabeleUZUP AS
    SELECT t.id, t.Lp, t.OSAD, t.IZ, t.Key, t.G, t.CzyPoprawna, t.ObroconyOSAD, m.Zgoda, m.Kobieta, m.Wiek, m.Wykształcenie, m.Zamieszkanie, m.inz, m.mat
    FROM tabeleUZUP AS t
    JOIN metryczkiABC AS m ON t.Key = m.Key
""")

In [18]:
con.execute("""
    SELECT *
    FROM tabeleUZUP
""").df()

,id,Lp,OSAD,IZ,Key,G,CzyPoprawna,ObroconyOSAD,Zgoda,Kobieta,Wiek,Wykształcenie,Zamieszkanie,inz,mat
0,0,1,2,1,B1,0,1,8,1,1,23,ś,500+,0,0
1,1,2,1,1,B1,0,1,9,1,1,23,ś,500+,0,0
2,2,3,4,1,B1,1,-1,6,1,1,23,ś,500+,0,0
3,4,5,2,1,B1,1,-1,8,1,1,23,ś,500+,0,0
4,5,6,7,1,B1,1,1,3,1,1,23,ś,500+,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1199,805,26,8,2,M9,1,1,2,1,1,19,s,w,0,1
1200,806,27,2,2,M9,0,1,8,1,1,19,s,w,0,1
1201,807,28,8,2,M9,1,1,2,1,1,19,s,w,0,1
1202,808,29,9,2,M9,0,-1,1,1,1,19,s,w,0,1


In [19]:
con.execute("""
    CREATE OR REPLACE TABLE ProcentPoprawnych AS
    SELECT * FROM (
    WITH poprawne AS
    (
        SELECT Key, COUNT(*) AS Poprawne
        FROM tabeleUZUP
        WHERE CzyPoprawna = 1
        GROUP BY Key
    ),
    niepoprawne AS
    (
        SELECT Key, COUNT(*) AS Niepoprawne
        FROM tabeleUZUP
        WHERE CzyPoprawna = -1
        GROUP BY Key
    )

    SELECT t.Key,
    p.Poprawne / (p.Poprawne + COALESCE(n.Niepoprawne,0)) * 100 AS ProcentPoprawnych,
    (SUM(t.IZ) - 28) / 28 AS ProcentZasad,
    t.mat, t.Wiek, t.Kobieta
    FROM tabeleUZUP as t
    LEFT JOIN poprawne AS p ON t.Key = p.Key
    LEFT JOIN niepoprawne AS n ON t.Key = n.Key
    GROUP BY t.Key, t.mat, t.Wiek, t.Kobieta, p.Poprawne, n.Niepoprawne
    ORDER BY t.Key
    )
""")



In [20]:
con.execute("""
    SELECT *
    FROM ProcentPoprawnych
""").df()

,Key,ProcentPoprawnych,ProcentZasad,mat,Wiek,Kobieta
0,A1,88.461538,0.392857,0,26,0
1,A10,92.592593,0.607143,0,23,1
2,A11,65.384615,0.535714,0,17,1
3,A2,74.074074,0.464286,0,23,1
4,A3,100.000000,0.928571,0,26,0
5,A4,89.285714,0.535714,0,26,1
6,A5,86.956522,0.571429,0,24,1
7,A6,81.481481,0.571429,0,21,1
8,A7,85.185185,0.500000,0,24,0
9,A8,85.185185,0.392857,0,23,1


In [21]:
con.execute("""
    CREATE OR REPLACE TABLE Odstajace AS
    WITH quantiles AS (
    SELECT
        quantile_cont(ProcentPoprawnych, 0.25) AS q25,
        quantile_cont(ProcentPoprawnych, 0.75) AS q75,
        quantile_cont(ProcentPoprawnych, 0.75) - quantile_cont(ProcentPoprawnych, 0.25) AS IQR
    FROM ProcentPoprawnych
    )

    SELECT pp.Key
    FROM ProcentPoprawnych pp
    CROSS JOIN quantiles q
    WHERE pp.ProcentPoprawnych < q.q25 - 1.5 * q.IQR
    OR pp.ProcentPoprawnych > q.q75 + 1.5 * q.IQR;

""")



In [22]:
con.execute("""
    SELECT *
    FROM Odstajace
""").df()

,Key
0,Do1


In [23]:
tabeleABCuzup = sprawdz_poprawnosc_odpowiedzi(tabeleABC, poprawneABC)
tabeleABCuzup = pd.merge(tabeleABCuzup, metryczkiABC, on="Key")
ProcentPoprwnychABC = policz_procenty(tabeleABCuzup)
OdstajaceABC = znajdz_odstajace_IQR(ProcentPoprwnychABC)
OdstajaceABC

Metoda odfiltrowania: IQR
Znalezione obserwacje odstające (Key): ['Do1']


['Do1']

In [24]:
tabeleXYZuzup = sprawdz_poprawnosc_odpowiedzi(tabeleXYZ, poprawneXYZ)
tabeleXYZuzup = pd.merge(tabeleXYZuzup, metryczkiXYZ, on="Key")
ProcentPoprwnychXYZ = policz_procenty(tabeleXYZuzup)
OdstajaceXYZ = znajdz_odstajace_IQR(ProcentPoprwnychXYZ)
OdstajaceXYZ

Metoda odfiltrowania: IQR
Znalezione obserwacje odstające (Key): ['S2']


['S2']

In [25]:
tabeleABC_odfiltrowane = tabeleABCuzup[~tabeleABCuzup['Key'].isin(OdstajaceABC)]
tabeleXYZ_odfiltrowane = tabeleXYZuzup[~tabeleXYZuzup['Key'].isin(OdstajaceXYZ)]

In [26]:
con.execute("""
    CREATE OR REPLACE TABLE tabele_odfiltrowane AS
    SELECT *
    FROM tabeleUZUP
    WHERE Key NOT IN (
        SELECT Key
        FROM Odstajace
    )
""")

In [27]:
con.execute("""
    SELECT *
    FROM tabele_odfiltrowane
""").df()

,id,Lp,OSAD,IZ,Key,G,CzyPoprawna,ObroconyOSAD,Zgoda,Kobieta,Wiek,Wykształcenie,Zamieszkanie,inz,mat
0,0,1,2,1,B1,0,1,8,1,1,23,ś,500+,0,0
1,1,2,1,1,B1,0,1,9,1,1,23,ś,500+,0,0
2,2,3,4,1,B1,1,-1,6,1,1,23,ś,500+,0,0
3,4,5,2,1,B1,1,-1,8,1,1,23,ś,500+,0,0
4,5,6,7,1,B1,1,1,3,1,1,23,ś,500+,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171,805,26,8,2,M9,1,1,2,1,1,19,s,w,0,1
1172,806,27,2,2,M9,0,1,8,1,1,19,s,w,0,1
1173,807,28,8,2,M9,1,1,2,1,1,19,s,w,0,1
1174,808,29,9,2,M9,0,-1,1,1,1,19,s,w,0,1


In [28]:
ABC = policz_procenty(tabeleABC_odfiltrowane)
XYZ = policz_procenty(tabeleXYZ_odfiltrowane)

In [29]:
con.execute("""
    CREATE OR REPLACE TABLE PP_odfiltrowane AS
    SELECT *
    FROM ProcentPoprawnych
    WHERE Key NOT IN (
        SELECT Key
        FROM Odstajace
    )
""")

In [30]:
con.execute("""
    SELECT *
    FROM PP_odfiltrowane
""").df()

,Key,ProcentPoprawnych,ProcentZasad,mat,Wiek,Kobieta
0,A1,88.461538,0.392857,0,26,0
1,A10,92.592593,0.607143,0,23,1
2,A11,65.384615,0.535714,0,17,1
3,A2,74.074074,0.464286,0,23,1
4,A3,100.000000,0.928571,0,26,0
5,A4,89.285714,0.535714,0,26,1
6,A5,86.956522,0.571429,0,24,1
7,A6,81.481481,0.571429,0,21,1
8,A7,85.185185,0.500000,0,24,0
9,A8,85.185185,0.392857,0,23,1


In [31]:
ABC

,Key,ProcentPoprawnych,ProcentZasad,mat,Wiek,Kobieta
0,A1,88.461538,39.285714,0,26,0
1,A10,92.592593,60.714286,0,23,1
2,A11,65.384615,53.571429,0,17,1
3,A2,74.074074,46.428571,0,23,1
4,A3,100.000000,92.857143,0,26,0
5,A4,89.285714,53.571429,0,26,1
6,A5,86.956522,57.142857,0,24,1
7,A6,81.481481,57.142857,0,21,1
8,A7,85.185185,50.000000,0,24,0
9,A8,85.185185,39.285714,0,23,1


In [32]:
All = pd.concat([ABC, XYZ], ignore_index=True)

In [33]:
con.execute("""
    SELECT MIN(Wiek), MAX(Wiek), AVG(Wiek), STDDEV(Wiek)
    FROM PP_odfiltrowane
""").df()

,min(Wiek),max(Wiek),avg(Wiek),stddev(Wiek)
0,17,55,24.166667,7.499322


In [34]:
All["Wiek"].min()

17

In [35]:
All["Wiek"].max()

55

In [36]:
All['Wiek'].mean()

np.float64(23.265060240963855)

In [37]:
All['Wiek'].std()

5.723473994830194

In [38]:
All['Kobieta'].sum()

np.int64(41)

In [39]:
len(All) - All['Kobieta'].sum()

np.int64(42)

##Przygotowanie danych pod analizę porównawczą matematyków i niematematyków


---



In [40]:
def grupowanie_po_wykształceniu(ABC, XYZ):
    # Dodanie sufiksu do kolumny "Key"
    ABC = ABC.copy()
    XYZ = XYZ.copy()

    ABC["Key"] = ABC["Key"].astype(str) + "_ABC"
    XYZ["Key"] = XYZ["Key"].astype(str) + "_XYZ"

    # Łączenie tabel
    df = pd.concat([ABC, XYZ], ignore_index=True)

    # Podział na "mat" i "niemat"
    mat = df[df["mat"] == 1].reset_index(drop=True)
    niemat = df[df["mat"] == 0].reset_index(drop=True)

    return mat, niemat

In [41]:
mat, niemat = grupowanie_po_wykształceniu(ABC,XYZ)

In [42]:
con.execute("""
    SELECT *, CONCAT(Key,'_ABC') AS KEY_ABC
    FROM PP_odfiltrowane
    WHERE mat = 1;
""").df()

,Key,ProcentPoprawnych,ProcentZasad,mat,Wiek,Kobieta,KEY_ABC
0,B2,76.923077,0.607143,1,27,0,B2_ABC
1,B3,85.714286,1.000000,1,25,1,B3_ABC
2,B4,88.461538,0.714286,1,26,0,B4_ABC
3,M1,82.142857,0.500000,1,20,0,M1_ABC
4,M10,85.714286,0.857143,1,19,0,M10_ABC
5,M11,83.333333,0.607143,1,20,1,M11_ABC
6,M12,96.428571,1.000000,1,22,0,M12_ABC
7,M13,96.428571,1.000000,1,22,0,M13_ABC
8,M14,88.461538,0.500000,1,21,0,M14_ABC
9,M15,83.333333,0.285714,1,20,0,M15_ABC


In [43]:
con.execute("""
    SELECT *, CONCAT(Key,'_ABC') AS KEY_ABC
    FROM PP_odfiltrowane
    WHERE mat = 0;
""").df()

,Key,ProcentPoprawnych,ProcentZasad,mat,Wiek,Kobieta,KEY_ABC
0,A1,88.461538,0.392857,0,26,0,A1_ABC
1,A10,92.592593,0.607143,0,23,1,A10_ABC
2,A11,65.384615,0.535714,0,17,1,A11_ABC
3,A2,74.074074,0.464286,0,23,1,A2_ABC
4,A3,100.000000,0.928571,0,26,0,A3_ABC
5,A4,89.285714,0.535714,0,26,1,A4_ABC
6,A5,86.956522,0.571429,0,24,1,A5_ABC
7,A6,81.481481,0.571429,0,21,1,A6_ABC
8,A7,85.185185,0.500000,0,24,0,A7_ABC
9,A8,85.185185,0.392857,0,23,1,A8_ABC


In [44]:
mat.to_excel(Output_path + '/mat.xlsx', index=False)
niemat.to_excel(Output_path + '/niemat.xlsx', index=False)

##Przygotowanie danych pod analizę porównawczą modeli


---



In [45]:
ABC.to_excel(Output_path + '/ABC.xlsx', index=False)
XYZ.to_excel(Output_path + '/XYZ.xlsx', index=False)

In [46]:
def wypisz_odpowiedzi_z_procentem(df1, df2):
    # Grupowanie danych po kolumnie "Lp."
    grouped = df1.groupby('Lp.')['CzyPoprawna'].value_counts().unstack(fill_value=0)

    # Dodanie kolumny "procent_poprawnych"
    grouped['procent_poprawnych'] = grouped[1] / (grouped[1] + grouped[-1]) * 100

    return grouped.merge(df2, on="Lp.")

In [47]:
con.execute("""
    CREATE OR REPLACE TABLE PoprawnoscPytan AS
    (
    SELECT Lp,
    COUNT(CASE WHEN CzyPoprawna = -1 THEN 1 END) AS "-1",
    COUNT(CASE WHEN CzyPoprawna = 0 THEN 1 END) "0",
    COUNT(CASE WHEN CzyPoprawna = 1 THEN 1 END) AS "1",
    COUNT(CASE WHEN CzyPoprawna = 1 THEN 1 END) / (COUNT(*) - COUNT(CASE WHEN CzyPoprawna = 0 THEN 1 END)) * 100 AS "ProcentPoprawnych",
    G
    FROM tabele_odfiltrowane
    GROUP BY Lp, G
    )
""")

In [48]:
con.execute("""
    SELECT *
    FROM PoprawnoscPytan
""").df()

,Lp,-1,0,1,ProcentPoprawnych,G
0,1,10,0,32,76.190476,0
1,2,10,1,31,75.609756,0
2,3,6,0,36,85.714286,1
3,5,15,2,25,62.500000,1
4,6,4,1,37,90.243902,1
5,7,6,1,35,85.365854,0
6,8,5,1,36,87.804878,0
7,9,7,3,32,82.051282,1
8,10,6,2,34,85.000000,0
9,11,6,1,35,85.365854,0


In [49]:
PoprawnoscPytanABC = wypisz_odpowiedzi_z_procentem(tabeleABC_odfiltrowane,poprawneABC)
PoprawnoscPytanXYZ = wypisz_odpowiedzi_z_procentem(tabeleXYZ_odfiltrowane,poprawneXYZ)

In [50]:
PoprawnoscPytanABC

,Lp.,-1,0,1,procent_poprawnych,G,Ciąg
0,1,10,0,32,76.190476,0,abccacb
1,2,10,1,31,75.609756,0,bacbcab
2,3,6,0,36,85.714286,1,abababcccab
3,5,15,2,25,62.500000,1,abcabcabcabcab
4,6,4,1,37,90.243902,1,abcababcab
5,7,6,1,35,85.365854,0,acbcabcab
6,8,5,1,36,87.804878,0,abcbcabccbcab
7,9,7,3,32,82.051282,1,abcabccab
8,10,6,2,34,85.000000,0,abcaabcaabcab
9,11,6,1,35,85.365854,0,aabcacbcaba


In [51]:
PoprawnoscPytanABC.to_excel(Output_path + '/PoprawnoscPytanABC.xlsx', index=False)
PoprawnoscPytanXYZ.to_excel(Output_path + '/PoprawnoscPytanXYZ.xlsx', index=False)

##Przygotowanie danych pod analizę pewności osądu gramatycznego
---



In [52]:
pewnoscABC = tabeleABC_odfiltrowane[["Lp.", "CzyPoprawna", "mat",  "obróconyOSAD", "OSAD", "G", "Key"]]
pewnoscXYZ = tabeleXYZ_odfiltrowane[["Lp.", "CzyPoprawna", "mat",  "obróconyOSAD", "OSAD", "G", "Key"]]

In [53]:
pewnoscABC

,Lp.,CzyPoprawna,mat,obróconyOSAD,OSAD,G,Key
0,1,1,0,8,2,0,B1
1,2,1,0,9,1,0,B1
2,3,-1,0,4,4,1,B1
3,5,-1,0,2,2,1,B1
4,6,1,0,7,7,1,B1
...,...,...,...,...,...,...,...
1199,26,1,0,8,8,1,O6
1200,27,1,0,8,2,0,O6
1201,28,1,0,6,6,1,O6
1202,29,-1,0,4,6,0,O6


In [54]:
con.execute("""
    SELECT Lp, CzyPoprawna, mat, obroconyOSAD, OSAD, G, Key
    FROM tabele_odfiltrowane
""").df()

,Lp,CzyPoprawna,mat,ObroconyOSAD,OSAD,G,Key
0,1,1,0,8,2,0,B1
1,2,1,0,9,1,0,B1
2,3,-1,0,6,4,1,B1
3,5,-1,0,8,2,1,B1
4,6,1,0,3,7,1,B1
...,...,...,...,...,...,...,...
1171,26,1,1,2,8,1,M9
1172,27,1,1,8,2,0,M9
1173,28,1,1,2,8,1,M9
1174,29,-1,1,1,9,0,M9


In [55]:
pewnoscABC.to_excel(Output_path + '/pewnoscABC.xlsx', index=False)
pewnoscXYZ.to_excel(Output_path + '/pewnoscXYZ.xlsx', index=False)

##Przygotowanie danych pod analizę odwoływania się do zasad i intuicji


---



In [56]:
zasadyABC = tabeleABC_odfiltrowane[["Lp.", "CzyPoprawna", "I/Z", "mat"]]
zasadyXYZ = tabeleXYZ_odfiltrowane[["Lp.", "CzyPoprawna", "I/Z", "mat"]]

In [57]:
zasadyABC

,Lp.,CzyPoprawna,I/Z,mat
0,1,1,1,0
1,2,1,1,0
2,3,-1,1,0
3,5,-1,1,0
4,6,1,1,0
...,...,...,...,...
1199,26,1,2,0
1200,27,1,2,0
1201,28,1,2,0
1202,29,-1,2,0


In [58]:
con.execute("""
    SELECT Lp, CzyPoprawna, IZ, mat
    FROM tabele_odfiltrowane
""").df()

,Lp,CzyPoprawna,IZ,mat
0,1,1,1,0
1,2,1,1,0
2,3,-1,1,0
3,5,-1,1,0
4,6,1,1,0
...,...,...,...,...
1171,26,1,2,1
1172,27,1,2,1
1173,28,1,2,1
1174,29,-1,2,1


In [59]:
zasadyABC.to_excel(Output_path + '/zasadyABC.xlsx', index=False)
zasadyXYZ.to_excel(Output_path + '/zasadyXYZ.xlsx', index=False)